**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../docs/METHODS.md) for the implemented protocol.


# Level 1 — Broad Taxonomic Classification (semi-supervised)

**Goal:** Demonstrate that graph SSL works on an easy 4-class problem
(Phocidae / Monodontidae / Balaenidae / Delphinidae subset). Run **both**
label propagation and Laplacian regularization, pick the winner, carry it to
Level 2.

Pipeline:
1. Load `Level1/Data/level1_features.npz`, standardize features.
2. Build symmetric k-NN graph (k=10, σ=median NN distance).
3. 50% labels, stratified, 20 random trials. Cross-validate μ for Laplacian
   regularization. Report mean ± std accuracy and macro F1.
4. Spectral gap analysis on the normalized Laplacian.
5. Diffusion map embedding + t-SNE side by side.
6. ResNet-18 baseline on Mel spectrograms, same 50% labeled subset.
7. Comparison table, confusion matrix, write all figures to `Level1/figures/`.


In [ ]:
import sys
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.graph import (
    build_knn_graph,
    compute_laplacian,
    spectral_gap_analysis,
    diffusion_map_embedding,
    label_propagation,
    laplacian_regularization,
)
from utils.evaluation import (
    classification_report_custom,
    plot_confusion_matrix,
    plot_spectral_gap,
    plot_embedding_2d,
)
from utils.cnn import train_resnet_baseline

LEVEL_DIR = PROJECT_ROOT / "Level1"
DATA_DIR = LEVEL_DIR / "Data"
FIG_DIR = LEVEL_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("project root:", PROJECT_ROOT)
print("data:", DATA_DIR)
print("figures:", FIG_DIR)


## Load features and standardize


In [ ]:
data = np.load(DATA_DIR / "level1_features.npz", allow_pickle=True)
X = data["X"].astype(np.float32)
y = data["labels"].astype(np.int64)
clip_ids = data["clip_ids"]
label_names = list(data["label_names"])
species = data["species"]
is_train = data["is_train"]
print(f"X: {X.shape}, labels: {y.shape}")
print("classes:", label_names)
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {label_names[u]}: {c}")

scaler = StandardScaler()
Xz = scaler.fit_transform(X).astype(np.float32)


## Build k-NN graph


In [ ]:
K_GRAPH = 10
W = build_knn_graph(Xz, k=K_GRAPH, sigma="median")
L_norm = compute_laplacian(W, normalized=True)
print(f"W: {W.shape}, nnz={W.nnz}, density={W.nnz / (W.shape[0]**2):.4f}")


## Spectral gap analysis

The first few eigenvalues of the random-walk Laplacian. A clear gap after the
4th eigenvalue would suggest the data partitions cleanly into 4 communities,
matching the 4 family groups.


In [ ]:
eigs = spectral_gap_analysis(L_norm, n_eigenvalues=10)
print("first 10 eigenvalues:", eigs)
plot_spectral_gap(eigs, FIG_DIR / "spectral_gap.png", title="Level 1 — Spectral gap")


## Stratified label-mask helper


In [ ]:
N_TRIALS = 20
LABEL_FRAC = 0.5

def stratified_mask(y, frac, rng):
    mask = np.zeros(y.shape[0], dtype=bool)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        n_lbl = max(1, int(round(frac * idx.size)))
        chosen = rng.choice(idx, size=n_lbl, replace=False)
        mask[chosen] = True
    return mask


## Cross-validate μ for Laplacian regularization

Within the **labeled** half of one trial, hold out 20% as validation, sweep μ,
pick the best. We do this once on a representative trial to save compute, then
reuse the chosen μ for all 20 trials.


In [ ]:
MU_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]
rng_cv = np.random.default_rng(123)
mask_cv = stratified_mask(y, LABEL_FRAC, rng_cv)
labeled_idx = np.where(mask_cv)[0]
val_size = max(4, int(0.2 * labeled_idx.size))
val_idx = rng_cv.choice(labeled_idx, size=val_size, replace=False)
train_mask = mask_cv.copy()
train_mask[val_idx] = False  # treat val nodes as unlabeled during fitting

mu_scores = {}
for mu in MU_GRID:
    pred = laplacian_regularization(W, y, train_mask, mu=mu)
    acc = accuracy_score(y[val_idx], pred[val_idx])
    mu_scores[mu] = acc
    print(f"  mu={mu:>6.2f}  val acc={acc:.4f}")
best_mu = max(mu_scores, key=mu_scores.get)
print(f"\nselected mu = {best_mu}")


## 20 random trials — both methods


In [ ]:
results = {"label_prop": {"acc": [], "f1": [], "preds": []},
           "lap_reg":    {"acc": [], "f1": [], "preds": []}}
trial_masks = []

t0 = time.time()
for trial in range(N_TRIALS):
    rng = np.random.default_rng(1000 + trial)
    mask = stratified_mask(y, LABEL_FRAC, rng)
    trial_masks.append(mask)
    unl = ~mask

    pred_lp = label_propagation(W, y, mask)
    pred_lr = laplacian_regularization(W, y, mask, mu=best_mu)

    acc_lp = accuracy_score(y[unl], pred_lp[unl])
    f1_lp = f1_score(y[unl], pred_lp[unl], average="macro", labels=range(len(label_names)), zero_division=0)
    acc_lr = accuracy_score(y[unl], pred_lr[unl])
    f1_lr = f1_score(y[unl], pred_lr[unl], average="macro", labels=range(len(label_names)), zero_division=0)

    results["label_prop"]["acc"].append(acc_lp)
    results["label_prop"]["f1"].append(f1_lp)
    results["label_prop"]["preds"].append(pred_lp)
    results["lap_reg"]["acc"].append(acc_lr)
    results["lap_reg"]["f1"].append(f1_lr)
    results["lap_reg"]["preds"].append(pred_lr)

    if (trial + 1) % 5 == 0:
        print(f"  trial {trial+1}/{N_TRIALS}: "
              f"LP acc={acc_lp:.3f}, LR acc={acc_lr:.3f}")
print(f"\n20-trial graph SSL took {time.time() - t0:.1f}s")

for name, r in results.items():
    a = np.array(r["acc"])
    f = np.array(r["f1"])
    print(f"  {name:>10s}: acc = {a.mean():.4f} ± {a.std():.4f}, "
          f"macro F1 = {f.mean():.4f} ± {f.std():.4f}")

WINNER = "label_prop" if np.mean(results["label_prop"]["acc"]) > np.mean(results["lap_reg"]["acc"]) else "lap_reg"
print(f"\n>>> Level 1 winner: {WINNER} <<<")


## Diffusion map embedding (primary) + t-SNE (comparison)


In [ ]:
diff_coords = diffusion_map_embedding(W, n_components=3, t=5)
print("diffusion coords:", diff_coords.shape)

tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=0)
tsne_coords = tsne.fit_transform(Xz)
print("t-SNE coords:", tsne_coords.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cmap = plt.cm.get_cmap("tab10", len(label_names))
for ax, coords, title in [(axes[0], diff_coords[:, :2], "Diffusion map (t=5)"),
                           (axes[1], tsne_coords, "t-SNE")]:
    for i, name in enumerate(label_names):
        m = y == i
        ax.scatter(coords[m, 0], coords[m, 1], s=12, alpha=0.7, color=cmap(i), label=name)
    ax.set_title(f"Level 1 — {title}")
    ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2")
    ax.legend(fontsize=8, markerscale=1.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "embeddings.png", dpi=150)
plt.close(fig)
print("saved embeddings.png")


## ResNet-18 baseline on Mel spectrograms

Trained on the **same 50% labeled subset** taken from clips that the
HuggingFace train split contains; evaluated on clips from the test split.
Uses the first trial's labeled mask for the train selection.


In [ ]:
spec = np.load(DATA_DIR / "level1_spectrograms.npz", allow_pickle=True)
S_all = spec["S"].astype(np.float32)
print("S:", S_all.shape)

mask0 = trial_masks[0]
train_keep = is_train & mask0
test_keep = ~is_train

S_tr = S_all[train_keep]
y_tr = y[train_keep]
S_te = S_all[test_keep]
y_te = y[test_keep]
print(f"CNN train (HF train ∩ labeled): {S_tr.shape[0]} windows")
print(f"CNN test  (HF test):             {S_te.shape[0]} windows")

cnn_out = train_resnet_baseline(
    S_tr, y_tr, S_te, y_te,
    num_classes=len(label_names),
    epochs=12, batch_size=32, lr=1e-3, seed=0, verbose=True,
)
cnn_acc = accuracy_score(cnn_out["y_true"], cnn_out["y_pred"])
cnn_f1 = f1_score(cnn_out["y_true"], cnn_out["y_pred"], average="macro",
                  labels=range(len(label_names)), zero_division=0)
print(f"\nResNet-18: acc={cnn_acc:.4f}, macro F1={cnn_f1:.4f}")


## Comparison table & confusion matrix for the winning graph method


In [ ]:
def fmt(arr): return f"{np.mean(arr):.4f} ± {np.std(arr):.4f}"

print("=== Level 1 results ===")
print(f"  Label propagation     : acc {fmt(results['label_prop']['acc'])}  macro F1 {fmt(results['label_prop']['f1'])}")
print(f"  Laplacian regulariz.  : acc {fmt(results['lap_reg']['acc'])}  macro F1 {fmt(results['lap_reg']['f1'])}")
print(f"  ResNet-18 baseline    : acc {cnn_acc:.4f}              macro F1 {cnn_f1:.4f}")
print(f"\n  Winner (graph SSL): {WINNER}")

best_pred = results[WINNER]["preds"][0]
unl0 = ~trial_masks[0]
plot_confusion_matrix(
    y[unl0], best_pred[unl0], label_names,
    save_path=FIG_DIR / f"confusion_matrix_{WINNER}.png",
    title=f"Level 1 — {WINNER} (trial 1, unlabeled nodes)",
)
plot_confusion_matrix(
    cnn_out["y_true"], cnn_out["y_pred"], label_names,
    save_path=FIG_DIR / "confusion_matrix_resnet.png",
    title="Level 1 — ResNet-18 (HF test split)",
)

# Save winner to disk so Level 2 can read it.
(LEVEL_DIR / "winner.txt").write_text(WINNER + "\n", encoding="utf-8")
print(f"\nwrote {LEVEL_DIR / 'winner.txt'}")
print("\nFigures saved to", FIG_DIR)
